# 0. Environment Setup

Mounts Google Drive, makes the shared modules in src/ importable, seeds the random number generators and creates the project folder structure.

After editing a module in src/, call reload_project_modules() and re-run the cells that import from it.

In [ ]:
# =====================================================
# 0.1 - BOOTSTRAP: DRIVE MOUNT AND MODULE PATH
# =====================================================
import importlib
import sys
from pathlib import Path

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/VineProject_ToF")
else:
    PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "src" / "config.py").exists():
    raise FileNotFoundError(f"config.py not found under: {PROJECT_ROOT / 'src'}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import config


def reload_project_modules() -> None:
    """Reload the modules in src/ after editing them.

    Python caches imported modules, so a module edited on Drive is not picked
    up by a simple re-import. Run this cell after uploading a new version of
    config.py or data_preparation.py, then re-run the cells that import from
    them.
    """
    # config first: the other modules read its constants at import time.
    for module_name in ("config", "data_preparation"):
        if module_name in sys.modules:
            importlib.reload(sys.modules[module_name])
            print(f"  reloaded: {module_name}")


config.print_environment_info()
config.set_seed()
config.create_project_structure()

Mounted at /content/drive
ENVIRONMENT INFO
Platform:       colab
Base path:      /content/drive/MyDrive/VineProject_ToF
Device:         cpu
PyTorch:        2.11.0+cpu
OpenCV:         5.0.0
Albumentations: 2.0.8
NumPy:          2.0.2
[OK] Random seed set to 42 (cuDNN deterministic: False)
Project structure ready at: /content/drive/MyDrive/VineProject_ToF
All folders already existed.


# 1. Label Map Generation

Converts the LabelMe JSON annotations into segmentation masks: a multiclass label map (5 classes) and a binary cordon/background map, plus their colored versions for visual inspection.

Label aliases, drawing priorities and ignored labels are defined in config.py, section 5. Set overwrite=True to rebuild everything from scratch.

Input: data/raw/annotations/*.json — Output: data/processed/labels/ and data/processed/colored_masks/

In [ ]:
# =====================================================
# 1.1 - GENERATE LABEL MAPS FROM LABELME ANNOTATIONS
# =====================================================
from data_preparation import generate_label_maps

generate_label_maps(overwrite=False)

LABEL MAPS GENERATION
Annotations:     /content/drive/MyDrive/VineProject_ToF/data/raw/annotations
Files to process:  95
Classes:         ['Background', 'Cane', 'Cordon', 'Node', 'Other']
Aliases:         {'tralcio': 'cane', 'tralcio_secco': 'cane', 'cordone': 'cordon', 'curva': 'cordon', 'nodo': 'node', 'altro': 'other'}
Ignored labels:  ['ferro', 'bud']
Priority multi:  ['cordon', 'node', 'other', 'cane']
Priority binary: ['cane', 'other', 'node', 'cordon']
Overwrite:       False


Generating label maps:  76%|███████▌  | 72/95 [00:27<00:08,  2.66it/s]

Generating label maps:  80%|████████  | 76/95 [00:28<00:06,  2.75it/s]

Generating label maps:  93%|█████████▎| 88/95 [00:33<00:02,  2.77it/s]

Generating label maps: 100%|██████████| 95/95 [00:35<00:00,  2.66it/s]

LABEL MAPS GENERATION COMPLETED
  Multiclass label maps: /content/drive/MyDrive/VineProject_ToF/data/processed/labels/multiclass
  Binary label maps:     /content/drive/MyDrive/VineProject_ToF/data/processed/labels/cordon_binary
  Colored multiclass:    /content/drive/MyDrive/VineProject_ToF/data/processed/colored_masks/multiclass
  Colored binary:        /content/drive/MyDrive/VineProject_ToF/data/processed/colored_masks/cordon_binary


# 2. Skeleton Map Generation

Precomputes the tubed skeleton of the elongated classes (Cane, Cordon), used by the clDice metric and by the skeleton recall loss. Running it offline keeps Zhang-Suen thinning out of the training loop, where it would dominate data loading time.

The cordon is skeletonised from the unified binary mask, so that canes crossing in front of it do not break the centreline; the occluded pixels are then removed from the result.

Skeleton classes and dilation kernel are set in config.SKELETON_LOSS_CONFIG.

Input: data/processed/labels/ — Output: data/processed/skeleton/ and data/processed/colored_masks/skeleton_overlay/

In [ ]:
# =====================================================
# 2.1 - GENERATE TUBED SKELETON MAPS
# =====================================================
from data_preparation import generate_skeleton_maps

generate_skeleton_maps(overwrite=False)